In [ ]:
# Run this only if a dependency is missing.
# %pip install -U --no-cache-dir gdown tabicl scikit-image scipy matplotlib numpy pandas scikit-learn


In [2]:
import os
import json
import tarfile
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.image import imread
from scipy.optimize import brentq
from sklearn.isotonic import IsotonicRegression
from skimage.transform import rescale, resize
from tabicl import TabICLRegressor

B_LOSS = 1.0


In [3]:
import gdown

# Load the cached tumor-segmentation data and PraNet probabilities.
# The .npz file is about 3.4 GB, so this cell can take a while.
# This works whether the notebook cwd is the repo root or Synthetic/.
cwd = Path.cwd()
project_root = cwd.parent if cwd.name == 'Synthetic' else cwd
data_dir = project_root / 'data'
archive_path = project_root / 'data.tar.gz'
npz_path = data_dir / 'polyps' / 'polyps-pranet.npz'

if not npz_path.exists():
    if not archive_path.exists():
        print('Downloading tumor-segmentation data archive...')
        gdown.download(id='1h7S6N_Rx7gdfO3ZunzErZy6H7620EbZK', output=str(archive_path), quiet=False)
    print('Extracting tumor-segmentation data archive...')
    with tarfile.open(archive_path, 'r:gz') as archive:
        archive.extractall(project_root)
    
print(f'Opening {npz_path}...')
data = np.load(npz_path)
example_paths = os.listdir(data_dir / 'polyps' / 'examples')

print('Loading probability maps and ground-truth masks into memory...')
probabilities = data['sgmd']       # PraNet sigmoid probabilities, shape: (1798, 352, 352)
gt_masks = data['targets']         # Ground-truth binary masks, shape: (1798, 352, 352)
example_indexes = data['example_indexes']

# Keep the original variable name from the upstream notebook too.
sgmd = probabilities
print('Loaded:', probabilities.shape, gt_masks.shape)

Opening /home/bresende/projects/pesquisa/CRC_rectification/data/polyps/polyps-pranet.npz...
Loading probability maps and ground-truth masks into memory...
Loaded: (1798, 352, 352) (1798, 352, 352)


In [4]:
# Route 2 first part: summarize each image by probability quantiles.
N_QUANTILES = 90
quantile_levels = np.linspace(0.0, 1.0, N_QUANTILES)
quantile_columns = [f"prob_q_{level:.3f}" for level in quantile_levels]

probability_quantiles = np.empty((probabilities.shape[0], N_QUANTILES), dtype=float)
for i, prob_map in enumerate(probabilities):
    if i % 100 == 0:
        print(f'Computing quantiles: {i}/{probabilities.shape[0]}')
    probability_quantiles[i] = np.quantile(prob_map.reshape(-1), quantile_levels)

X_quantiles = pd.DataFrame(probability_quantiles, columns=quantile_columns)
X_quantiles.head()

Computing quantiles: 0/1798
Computing quantiles: 100/1798
Computing quantiles: 200/1798
Computing quantiles: 300/1798
Computing quantiles: 400/1798
Computing quantiles: 500/1798
Computing quantiles: 600/1798
Computing quantiles: 700/1798
Computing quantiles: 800/1798
Computing quantiles: 900/1798
Computing quantiles: 1000/1798
Computing quantiles: 1100/1798
Computing quantiles: 1200/1798
Computing quantiles: 1300/1798
Computing quantiles: 1400/1798
Computing quantiles: 1500/1798
Computing quantiles: 1600/1798
Computing quantiles: 1700/1798


,prob_q_0.000,prob_q_0.011,prob_q_0.022,prob_q_0.034,prob_q_0.045,prob_q_0.056,prob_q_0.067,prob_q_0.079,prob_q_0.090,prob_q_0.101,...,prob_q_0.899,prob_q_0.910,prob_q_0.921,prob_q_0.933,prob_q_0.944,prob_q_0.955,prob_q_0.966,prob_q_0.978,prob_q_0.989,prob_q_1.000
0,0.050792,0.073191,0.081550,0.092657,0.106634,0.121850,0.136465,0.148961,0.160722,0.170434,...,0.933368,0.940079,0.946691,0.952015,0.956770,0.964139,0.969930,0.976229,0.984852,0.993642
1,0.045520,0.078742,0.096350,0.108779,0.115703,0.122693,0.129104,0.136347,0.143960,0.151150,...,0.961271,0.965899,0.969553,0.973213,0.977297,0.980947,0.984069,0.986506,0.988758,0.991239
2,0.019697,0.044503,0.058211,0.068318,0.077653,0.086320,0.095590,0.105390,0.115517,0.125297,...,0.949570,0.954652,0.959128,0.962979,0.966165,0.969282,0.972605,0.976466,0.982414,0.992084
3,0.020888,0.044716,0.078553,0.105011,0.118950,0.127722,0.135111,0.141866,0.148176,0.155107,...,0.606314,0.706422,0.772786,0.814167,0.847059,0.875353,0.898734,0.917679,0.949707,0.984163
4,0.009730,0.028659,0.039962,0.052520,0.060882,0.070393,0.081713,0.093708,0.106477,0.118845,...,0.289514,0.298833,0.410957,0.599275,0.774825,0.886929,0.952223,0.983485,0.995391,0.999208


In [5]:
def segmentation_mask(probability_maps, lam):
    """
    Protective-lambda convention:
        C_lambda(x) = {pixels: p(pixel) >= 1 - lambda}.

    This is equivalent to the PDF threshold convention with
        threshold = 1 - lambda.

    Larger lambda lowers the threshold, gives a larger predicted mask,
    and therefore produces lower or equal false-negative risk.
    """
    lam = np.asarray(lam)
    if lam.ndim == 1:
        lam = lam[:, None, None]
    threshold = 1.0 - lam
    return probability_maps >= threshold


def false_negative_rate(probability_maps, true_masks, lam):
    """
    Per-image false negative rate:
        missed true tumor pixels / true tumor pixels.
    """
    pred_masks = segmentation_mask(probability_maps, lam)
    true_masks_bool = true_masks.astype(bool)

    n_positive = true_masks_bool.reshape(true_masks_bool.shape[0], -1).sum(axis=1)
    missed_positive = (true_masks_bool & ~pred_masks).reshape(true_masks_bool.shape[0], -1).sum(axis=1)

    return np.divide(
        missed_positive,
        n_positive,
        out=np.zeros_like(missed_positive, dtype=float),
        where=n_positive > 0,
    )


def build_route2_augmented_dataset(probability_maps, true_masks, image_features, K_aug=20, rng=None):
    """
    Build Route 2 augmented data:
        (90 probability quantiles, lambda) -> false negative rate.

    For each image, draw K_aug lambdas independently from Uniform(0, 1).
    """
    if rng is None:
        rng = np.random.default_rng()

    n_images = probability_maps.shape[0]
    lambda_draws = rng.uniform(0.0, 1.0, size=(n_images, K_aug))

    X_rep = np.repeat(image_features.to_numpy(), K_aug, axis=0)
    lambda_rep = lambda_draws.reshape(-1)

    losses = np.empty(n_images * K_aug, dtype=float)
    for k in range(K_aug):
        losses[k::K_aug] = false_negative_rate(
            probability_maps,
            true_masks,
            lambda_draws[:, k][:, None, None],
        )

    X_aug = pd.DataFrame(X_rep, columns=image_features.columns)
    X_aug["lambda"] = lambda_rep

    y_aug = pd.Series(losses, name="false_negative_rate")

    return X_aug, y_aug, lambda_draws


def inverter_R(R_row, lambda_grid, a):
    """inf {lambda in grid: estimated R(lambda | x) <= a}."""
    mask = R_row <= a
    if not mask.any():
        return float(lambda_grid[-1])
    return float(lambda_grid[np.argmax(mask)])


def lambdas_from_risk_matrix(R_mat, lambda_grid, a):
    """Apply inverter_R row-by-row to get one lambda per image."""
    return np.array([inverter_R(row, lambda_grid, a) for row in R_mat], dtype=float)


def calibrar_crc(losses_cal, lambda_grid, alpha, B=B_LOSS):
    """Standard CRC: smallest global lambda with corrected risk <= alpha."""
    n = losses_cal.shape[0]
    R_cal = losses_cal.mean(axis=0)
    R_upper = (n * R_cal + B) / (n + 1)
    mask = R_upper <= alpha
    if not mask.any():
        return float(lambda_grid[-1])
    return float(lambda_grid[np.argmax(mask)])


def calibrar_crc_rectified(losses_rect, a_grid, alpha, B=B_LOSS):
    """Rectified CRC: largest risk budget a with corrected risk <= alpha."""
    n = losses_rect.shape[0]
    R_cal = losses_rect.mean(axis=0)
    R_upper = (n * R_cal + B) / (n + 1)
    mask = R_upper <= alpha
    if not mask.any():
        return float(a_grid[0])
    return float(a_grid[np.where(mask)[0].max()])


def fit_risk_surface_tabicl(probability_maps, true_masks, image_features, K_aug=15,
                            device="cuda", rng=None):
    """Fit TabICL to estimate the conditional risk surface."""
    X_aug, y_aug, _ = build_route2_augmented_dataset(
        probability_maps,
        true_masks,
        image_features,
        K_aug=K_aug,
        rng=rng,
    )
    reg = TabICLRegressor(device=device, kv_cache=True, random_state=42)
    reg.fit(X_aug, y_aug)
    return reg


def estimate_R_route2(reg_surface, image_features, lambda_grid):
    """
    Estimate R(lambda | x) on a grid of lambda values using TabICL.

    Returns an n x L matrix and enforces nonincreasing risk in lambda
    by reverse isotonic regression for each queried x.
    """
    if isinstance(image_features, pd.DataFrame):
        X_base = image_features.reset_index(drop=True).copy()
    else:
        X_base = pd.DataFrame(np.asarray(image_features))

    n = len(X_base)
    L = len(lambda_grid)
    R_hat = np.empty((n, L))

    for j, lam in enumerate(lambda_grid):
        X_query = X_base.copy()
        X_query["lambda"] = lam
        R_hat[:, j] = reg_surface.predict(X_query)

    R_hat = np.clip(R_hat, 0.0, B_LOSS)

    for i in range(n):
        iso = IsotonicRegression(increasing=True, out_of_bounds="clip")
        neg_R_iso = iso.fit_transform(lambda_grid, -R_hat[i])
        R_hat[i] = np.clip(-neg_R_iso, 0.0, B_LOSS)

    return R_hat


def losses_over_lambda_segmentation(probability_maps, true_masks, lambda_grid):
    """Calibration/test FNR matrix with shape n_images x len(lambda_grid)."""
    losses = np.empty((probability_maps.shape[0], len(lambda_grid)), dtype=float)
    for j, lam in enumerate(lambda_grid):
        losses[:, j] = false_negative_rate(probability_maps, true_masks, lam)
    return losses


def calibrate_rectified_segmentation(probability_maps_cal, true_masks_cal,
                                     R_cal_mat, lambda_grid, a_grid,
                                     alpha, B=B_LOSS):
    """
    Choose a_hat using the calibration images.

    For each candidate a, invert each calibration image's estimated risk curve
    to get lambda_i(a), compute the true calibration FNR at that lambda_i(a),
    and apply the CRC finite-sample correction.
    """
    losses_rect = np.empty((probability_maps_cal.shape[0], len(a_grid)), dtype=float)

    for j, a in enumerate(a_grid):
        lambda_cal = lambdas_from_risk_matrix(R_cal_mat, lambda_grid, a)
        losses_rect[:, j] = false_negative_rate(
            probability_maps_cal,
            true_masks_cal,
            lambda_cal,
        )

    a_hat = calibrar_crc_rectified(losses_rect, a_grid, alpha, B=B)
    n = losses_rect.shape[0]
    risk = losses_rect.mean(axis=0)
    risk_upper = (n * risk + B) / (n + 1)
    calibration_curve = pd.DataFrame({
        "a": a_grid,
        "risk": risk,
        "risk_upper": risk_upper,
    })

    return a_hat, losses_rect, calibration_curve


def rectified_lambdas_and_masks(reg_surface, image_features, probability_maps,
                                lambda_grid, a_hat):
    """Estimate R(lambda | x), invert at a_hat, and return masks."""
    R_mat = estimate_R_route2(reg_surface, image_features, lambda_grid)
    lambda_vec = lambdas_from_risk_matrix(R_mat, lambda_grid, a_hat)
    masks = segmentation_mask(probability_maps, lambda_vec)
    return lambda_vec, masks, R_mat


def fit_calibrate_rectified_route2_segmentation(
    probability_maps_context,
    true_masks_context,
    features_context,
    probability_maps_cal,
    true_masks_cal,
    features_cal,
    probability_maps_target,
    features_target,
    alpha=0.10,
    lambda_grid=None,
    a_grid=None,
    K_aug=20,
    device="cuda",
    rng=None,
):
    """
    Full segmentation Route 2 pipeline:
      1. fit TabICL risk surface on context images;
      2. estimate R(lambda | x) on calibration images;
      3. calibrate a_hat by rectified CRC;
      4. estimate one lambda per target image and create masks.
    """
    if lambda_grid is None:
        lambda_grid = np.linspace(0.0, 1.0, 51)
    if a_grid is None:
        a_grid = np.linspace(0.001, 1.0, 50)

    reg_surface = fit_risk_surface_tabicl(
        probability_maps_context,
        true_masks_context,
        features_context,
        K_aug=K_aug,
        device=device,
        rng=rng,
    )

    R_cal_mat = estimate_R_route2(reg_surface, features_cal, lambda_grid)
    a_hat, losses_rect_cal, calibration_curve = calibrate_rectified_segmentation(
        probability_maps_cal,
        true_masks_cal,
        R_cal_mat,
        lambda_grid,
        a_grid,
        alpha,
        B=B_LOSS,
    )

    lambda_target, target_masks, R_target_mat = rectified_lambdas_and_masks(
        reg_surface,
        features_target,
        probability_maps_target,
        lambda_grid,
        a_hat,
    )

    return {
        "reg_surface": reg_surface,
        "a_hat": a_hat,
        "lambda_target": lambda_target,
        "target_masks": target_masks,
        "R_cal_mat": R_cal_mat,
        "R_target_mat": R_target_mat,
        "losses_rect_cal": losses_rect_cal,
        "calibration_curve": calibration_curve,
        "lambda_grid": lambda_grid,
        "a_grid": a_grid,
    }

In [6]:
rng = np.random.default_rng(2026)

X_aug, y_aug, lambda_draws = build_route2_augmented_dataset(
    probability_maps=probabilities,
    true_masks=gt_masks,
    image_features=X_quantiles,
    K_aug=20,
    rng=rng,
)

print("10 lambda draws for the first image:")
print(np.round(lambda_draws[0], 4))
print("X_aug shape:", X_aug.shape)
print("y_aug shape:", y_aug.shape)

display(X_aug.head())
display(y_aug.head())

10 lambda draws for the first image:
[0.1789 0.6399 0.4673 0.3705 0.3549 0.7905 0.9051 0.1774 0.6528 0.2983
 0.967  0.9199 0.6359 0.7527 0.5152 0.8259 0.4484 0.3388 0.2779 0.2263]
X_aug shape: (35960, 91)
y_aug shape: (35960,)


,prob_q_0.000,prob_q_0.011,prob_q_0.022,prob_q_0.034,prob_q_0.045,prob_q_0.056,prob_q_0.067,prob_q_0.079,prob_q_0.090,prob_q_0.101,...,prob_q_0.910,prob_q_0.921,prob_q_0.933,prob_q_0.944,prob_q_0.955,prob_q_0.966,prob_q_0.978,prob_q_0.989,prob_q_1.000,lambda
0,0.050792,0.073191,0.08155,0.092657,0.106634,0.12185,0.136465,0.148961,0.160722,0.170434,...,0.940079,0.946691,0.952015,0.95677,0.964139,0.96993,0.976229,0.984852,0.993642,0.178935
1,0.050792,0.073191,0.08155,0.092657,0.106634,0.12185,0.136465,0.148961,0.160722,0.170434,...,0.940079,0.946691,0.952015,0.95677,0.964139,0.96993,0.976229,0.984852,0.993642,0.639913
2,0.050792,0.073191,0.08155,0.092657,0.106634,0.12185,0.136465,0.148961,0.160722,0.170434,...,0.940079,0.946691,0.952015,0.95677,0.964139,0.96993,0.976229,0.984852,0.993642,0.467268
3,0.050792,0.073191,0.08155,0.092657,0.106634,0.12185,0.136465,0.148961,0.160722,0.170434,...,0.940079,0.946691,0.952015,0.95677,0.964139,0.96993,0.976229,0.984852,0.993642,0.370501
4,0.050792,0.073191,0.08155,0.092657,0.106634,0.12185,0.136465,0.148961,0.160722,0.170434,...,0.940079,0.946691,0.952015,0.95677,0.964139,0.96993,0.976229,0.984852,0.993642,0.354917


0    0.312730
1    0.000656
2    0.026539
3    0.085495
4    0.097195
Name: false_negative_rate, dtype: float64

In [ ]:
import time
import torch

# Full Route 2 segmentation run.
# Start with these sizes; increase them after the pipeline runs end-to-end.
ALPHA = 0.10
N_CONTEXT = 500
N_CAL = 300
K_AUG = 20
lambda_grid = np.linspace(0.0, 1.0, 51)
a_grid = np.linspace(0.001, 1.0, 50)

rng_route2 = np.random.default_rng(2026)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("TabICL device:", device)

# We will compute lambdas for the images that have example .jpg files.
target_indexes = np.asarray(example_indexes, dtype=int)
all_indexes = np.arange(probabilities.shape[0])

# Keep context/calibration independent from the displayed target examples.
pool_indexes = np.setdiff1d(all_indexes, target_indexes)
pool_indexes = rng_route2.permutation(pool_indexes)
context_indexes = pool_indexes[:N_CONTEXT]
cal_indexes = pool_indexes[N_CONTEXT:N_CONTEXT + N_CAL]

if len(context_indexes) < N_CONTEXT or len(cal_indexes) < N_CAL:
    raise ValueError("Not enough non-example images for the requested context/calibration sizes.")

print("context/calibration/target sizes:", len(context_indexes), len(cal_indexes), len(target_indexes))

t0 = time.time()
route2_out = fit_calibrate_rectified_route2_segmentation(
    probability_maps_context=probabilities[context_indexes],
    true_masks_context=gt_masks[context_indexes],
    features_context=X_quantiles.iloc[context_indexes].reset_index(drop=True),
    probability_maps_cal=probabilities[cal_indexes],
    true_masks_cal=gt_masks[cal_indexes],
    features_cal=X_quantiles.iloc[cal_indexes].reset_index(drop=True),
    probability_maps_target=probabilities[target_indexes],
    features_target=X_quantiles.iloc[target_indexes].reset_index(drop=True),
    alpha=ALPHA,
    lambda_grid=lambda_grid,
    a_grid=a_grid,
    K_aug=K_AUG,
    device=device,
    rng=rng_route2,
)

lambda_by_image_index = np.full(probabilities.shape[0], np.nan, dtype=float)
lambda_by_image_index[target_indexes] = route2_out["lambda_target"]
predicted_masks_by_target = route2_out["target_masks"]

print(f"finished in {time.time() - t0:.1f}s")
print("a_hat:", route2_out["a_hat"])
print(pd.Series(route2_out["lambda_target"]).describe())
display(route2_out["calibration_curve"].head())


In [ ]:
# Show examples using each image's own rectified lambda.
rng_examples = np.random.default_rng(7)
example_dir = data_dir / "polyps" / "examples"

for i in range(10):
    rand_idx = int(rng_examples.choice(target_indexes))
    lam_i = lambda_by_image_index[rand_idx]

    img = imread(example_dir / f"{rand_idx}.jpg")
    gt_mask = imread(example_dir / f"{rand_idx}_gt_mask.jpg")

    predicted_mask_352 = segmentation_mask(
        sgmd[rand_idx:rand_idx + 1],
        np.array([lam_i]),
    )[0]
    predicted_mask = resize(
        predicted_mask_352.astype(float),
        (img.shape[0], img.shape[1]),
        order=0,
        anti_aliasing=False,
        preserve_range=True,
    ) > 0.5

    fig, axs = plt.subplots(1, 3, figsize=(8.64, 4.76))
    axs[0].imshow(img)
    axs[0].axis("off")
    axs[1].imshow(predicted_mask, cmap="gray")
    axs[1].axis("off")
    axs[2].imshow(gt_mask, cmap="gray")
    axs[2].axis("off")

    if i == 0:
        axs[0].set_title("input")
        axs[1].set_title("predicted mask")
        axs[2].set_title("ground truth mask")

    fig.suptitle(f"image {rand_idx} | lambda={lam_i:.3f} | threshold={1 - lam_i:.3f}")
    plt.tight_layout()
    plt.show()
